In [1]:
import pandas as pd

In [3]:
import pandas as pd

# Load both CSV files
df1 = pd.read_csv('dis.csv')
df2 = pd.read_csv('egy.csv')

print("=== FILE 1: dis.csv ===")
print("Rows:", df1.shape[0], "| Columns:", df1.shape[1])
print("Features:", df1.columns.tolist())

print("\n=== FILE 2: egy.csv ===")
print("Rows:", df2.shape[0], "| Columns:", df2.shape[1])
print("Features:", df2.columns.tolist())

=== FILE 1: dis.csv ===
Rows: 29 | Columns: 3
Features: ['Feature Names', 'Feature Values', 'Discretization (Items)']

=== FILE 2: egy.csv ===
Rows: 1385 | Columns: 29
Features: ['Age ', 'Gender', 'BMI', 'Fever', 'Nausea/Vomting', 'Headache ', 'Diarrhea ', 'Fatigue & generalized bone ache ', 'Jaundice ', 'Epigastric pain ', 'WBC', 'RBC', 'HGB', 'Plat', 'AST 1', 'ALT 1', 'ALT4', 'ALT 12', 'ALT 24', 'ALT 36', 'ALT 48', 'ALT after 24 w', 'RNA Base', 'RNA 4', 'RNA 12', 'RNA EOT', 'RNA EF', 'Baseline histological Grading', 'Baselinehistological staging']


In [7]:
!pip install scikit-learn pandas numpy matplotlib seaborn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 139.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 112.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


In [9]:
import pandas as pd
df = pd.read_csv('egy.csv')
print(df.columns.tolist())

['Age ', 'Gender', 'BMI', 'Fever', 'Nausea/Vomting', 'Headache ', 'Diarrhea ', 'Fatigue & generalized bone ache ', 'Jaundice ', 'Epigastric pain ', 'WBC', 'RBC', 'HGB', 'Plat', 'AST 1', 'ALT 1', 'ALT4', 'ALT 12', 'ALT 24', 'ALT 36', 'ALT 48', 'ALT after 24 w', 'RNA Base', 'RNA 4', 'RNA 12', 'RNA EOT', 'RNA EF', 'Baseline histological Grading', 'Baselinehistological staging']


In [11]:
# Change 'target_column_name' to your actual target column name from above
target_col = 'Baseline histological staging'

In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load dataset
df = pd.read_csv('egy.csv')

# 2. Separate features (X) and target (y)
target_col = 'YOUR_ACTUAL_TARGET_COLUMN_NAME' # Replace with actual column name

X = df.drop(columns=[target_col])
y = df[target_col]

# 3. Split dataset (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Save test_data.csv for Streamlit upload
test_data = X_test.copy()
test_data[target_col] = y_test
test_data.to_csv('test_data.csv', index=False)

print("test_data.csv successfully saved with shape:", test_data.shape)

/home/cloud/anaconda3/lib/python3.12/site-packages/scipy/sparse/_sputils.py:17: FutureWarning: In the future `np.long` will be defined as the corresponding NumPy scalar.
  np.uintc, np.long, np.ulong, np.longlong, np.ulonglong,


AttributeError: module 'numpy' has no attribute 'long'

In [3]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, 
    recall_score, f1_score, matthews_corrcoef
)

# 1. Load dataset and clean column names by stripping spaces
df = pd.read_csv('egy.csv')
df.columns = df.columns.str.strip()

# 2. Dynamically match the target column name
target_candidates = [col for col in df.columns if 'histological staging' in col.lower()]
if target_candidates:
    target_col = target_candidates[0]
else:
    target_col = df.columns[-1]  # Fallback to last column

print(f"Target column set to: '{target_col}'")

X = df.drop(columns=[target_col])
y = df[target_col]

# Handle any non-numeric feature columns
X = pd.get_dummies(X, drop_first=True)

# 3. Split into Train (80%) and Test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Save test_data.csv for Streamlit deployment
test_data = X_test.copy()
test_data[target_col] = y_test
test_data.to_csv('test_data.csv', index=False)
print("Saved test_data.csv with shape:", test_data.shape)

# 5. Create directory to save binary model files
os.makedirs('model', exist_ok=True)

# 6. Train all 5 required models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'kNN': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
    'Support Vector Machine': SVC(probability=True, random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    
    # Save trained model object to model/ directory
    file_name = name.lower().replace(' ', '_') + '.pkl'
    joblib.dump(model, os.path.join('model', file_name))
    
    y_pred = model.predict(X_test)
    
    # Calculate multi-class AUC
    try:
        y_prob = model.predict_proba(X_test)
        auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
    except Exception:
        auc = "N/A"
        
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    mcc = matthews_corrcoef(y_test, y_pred)
    
    results.append({
        'ML Model Name': name,
        'Accuracy': round(acc, 4),
        'AUC': round(auc, 4) if isinstance(auc, float) else auc,
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4),
        'MCC': round(mcc, 4)
    })

# Output formatted Markdown comparison table for README.md
comparison_df = pd.DataFrame(results)
print("\n=== MODEL COMPARISON TABLE FOR README.md ===")
print(comparison_df.to_markdown(index=False))

Target column set to: 'Baselinehistological staging'
Saved test_data.csv with shape: (277, 29)


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



=== MODEL COMPARISON TABLE FOR README.md ===
| ML Model Name          |   Accuracy |    AUC |   Precision |   Recall |     F1 |     MCC |
|:-----------------------|-----------:|-------:|------------:|---------:|-------:|--------:|
| Logistic Regression    |     0.3141 | 0.5191 |      0.3315 |   0.3141 | 0.298  |  0.0853 |
| Decision Tree          |     0.213  | 0.4741 |      0.2105 |   0.213  | 0.2092 | -0.0515 |
| kNN                    |     0.2094 | 0.4916 |      0.221  |   0.2094 | 0.2095 | -0.0527 |
| Naive Bayes            |     0.2671 | 0.499  |      0.257  |   0.2671 | 0.2508 |  0.0207 |
| Support Vector Machine |     0.2635 | 0.5132 |      0.178  |   0.2635 | 0.1726 |  0.01   |


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [7]:
import os

model_dir = 'model'

if os.path.exists(model_dir):
    files = os.listdir(model_dir)
    print(f"Directory '{model_dir}' found!")
    print("Generated files:", files)
    
    expected_models = [
        'logistic_regression.pkl',
        'decision_tree.pkl',
        'knn.pkl',
        'naive_bayes.pkl',
        'support_vector_machine.pkl'
    ]
    
    print("\n--- Model Verification ---")
    for model_file in expected_models:
        file_path = os.path.join(model_dir, model_file)
        if os.path.exists(file_path):
            size_kb = os.path.getsize(file_path) / 1024
            print(f"✓ {model_file} exists ({size_kb:.2f} KB)")
        else:
            print(f"✗ {model_file} is MISSING!")
else:
    print(f"Directory '{model_dir}' does NOT exist yet. Please run your training script.")

Directory 'model' found!
Generated files: ['logistic_regression.pkl', 'decision_tree.pkl', 'knn.pkl', 'naive_bayes.pkl', 'support_vector_machine.pkl', '.ipynb_checkpoints']

--- Model Verification ---
✓ logistic_regression.pkl exists (2.44 KB)
✓ decision_tree.pkl exists (71.27 KB)
✓ knn.pkl exists (252.54 KB)
✓ naive_bayes.pkl exists (3.26 KB)
✓ support_vector_machine.pkl exists (301.22 KB)
